In [76]:
import pandas as pd
import numpy as np


In [77]:
df_task1 = pd.read_csv('../data/raw/ielts_writing_dataset.csv')
df2_task1 = pd.read_csv('../data/raw/nlpatunt_D_Ielts_Writing_Dataset_all.csv')
df3_task1 = pd.read_csv('../data/raw/TraTacXiMuoi_Ielts_writing_task1_academic_all.csv')

In [ ]:
def resolve_scores(scores):
    scores = [s for s in scores if pd.notna(s)]
    
    if len(scores) == 0:
        return np.nan
    
    if max(scores) - min(scores) >= 1.5:
        return None
    
    return np.median(scores)


def round_band(x):
    if pd.isna(x):
        return np.nan

    integer = int(x)
    decimal = x - integer

    if decimal < 0.25:
        return integer
    elif decimal <= 0.5:
        return integer + 0.5
    elif decimal < 0.75:
        return integer + 0.5
    else:
        return integer + 1


In [79]:
GRAPH_KEYWORDS = {
    "Bar Chart": ["bar chart", "bar graph", "bar"],
    "Line Graph": ["line graph", "line chart", "trend line"],
    "Pie Chart": ["pie chart", "pie graph"],
    "Table": ["table"],
    "Map": ["map"],
    "Process Diagram": ["process", "diagram", "flow chart", "flowchart"]
}


def extract_graph_types(text):
    if pd.isna(text):
        return set()
    
    text = text.lower()
    found = set()

    for graph_type in GRAPH_KEYWORDS:
        keywords = GRAPH_KEYWORDS[graph_type]
        for kw in keywords:
            if kw in text:
                found.add(graph_type)
    
    return found


def resolve_topic(question, essay):
    q_types = extract_graph_types(question)

    if len(q_types) == 0:
        types = extract_graph_types(essay)
    else:
        types = q_types

    if len(types) == 0:
        return np.nan

    if "Process Diagram" in types:
        return "Process Diagram"

    if len(types) >= 2:
        return "Multiple Graphs"

    return list(types)[0]

In [80]:
# Split
df1_task1 = df_task1[df_task1["Task_Type"] == 1].copy()
df2_task1_split = df2_task1[df2_task1["Task_Type"] == 1].copy()

In [81]:
df1_task1_clean = pd.DataFrame({
    "Topic": np.nan,
    "Question": df1_task1["Question"],
    "Essay": df1_task1["Essay"],
    "Task_Response": df1_task1["Task_Response"],
    "Coherence_Cohesion": df1_task1["Coherence_Cohesion"],
    "Lexical_Resource": df1_task1["Lexical_Resource"],
    "Range_Accuracy": df1_task1["Range_Accuracy"],
    "Overall": df1_task1["Overall"],
    "Examiner_Comment": df1_task1["Examiner_Commen"]
})

df2_task1_clean = pd.DataFrame({
    "Topic": np.nan,
    "Question": df2_task1_split["Question"],
    "Essay": df2_task1_split["Essay"],
    "Task_Response": np.nan,
    "Coherence_Cohesion": np.nan,
    "Lexical_Resource": np.nan,
    "Range_Accuracy": np.nan,
    "Overall": df2_task1_split["Overall_Score"],
    "Examiner_Comment": df2_task1_split["Examiner_Comments"]
})

df3_task1_clean = pd.DataFrame({
    "Topic": df3_task1["topic"],
    "Question": df3_task1["subject"],
    "Essay": df3_task1["content"],
    "Task_Response": np.nan,
    "Coherence_Cohesion": np.nan,
    "Lexical_Resource": np.nan,
    "Range_Accuracy": np.nan,
    "Overall": df3_task1["overall_band_score"],
    "Examiner_Comment": df3_task1["evaluation"]
})

In [82]:
# Merge
df_task1_final = pd.concat([
    df1_task1_clean,
    df2_task1_clean,
    df3_task1_clean
], ignore_index=True)

print("Raw size:", len(df_task1_final))

Raw size: 15209


In [83]:
# Remove essays in multiple questions
essay_q_count = df_task1_final.groupby("Essay")["Question"].nunique()
conflict_essays = essay_q_count[essay_q_count > 1].index

df_task1_final = df_task1_final[
    ~df_task1_final["Essay"].isin(conflict_essays)
]

print("Removed cross-question essays:", len(conflict_essays))


Removed cross-question essays: 10


In [84]:
# Aggregate duplicate essays
grouped1 = df_task1_final.groupby(["Question", "Essay"]).agg({
    "Overall": list,
    "Topic": "first",
    "Task_Response": "first",
    "Coherence_Cohesion": "first",
    "Lexical_Resource": "first",
    "Range_Accuracy": "first",
    "Examiner_Comment": "first"
}).reset_index()

In [85]:
grouped1["Overall_clean"] = grouped1["Overall"].apply(resolve_scores)

before = len(grouped1)
grouped1 = grouped1.dropna(subset=["Overall_clean"])
after = len(grouped1)

print("Removed noisy duplicates:", before - after)

Removed noisy duplicates: 9


In [86]:
# Finalize scores
df_task1_final = grouped1.drop(columns=["Overall"])
df_task1_final = df_task1_final.rename(columns={"Overall_clean": "Overall"})
df_task1_final["Overall"] = df_task1_final["Overall"].apply(round_band)


In [87]:
topics = []
for i in range(len(df_task1_final)):
    row = df_task1_final.iloc[i]
    topic = resolve_topic(row["Question"], row["Essay"])
    topics.append(topic)

df_task1_final["Topic"] = topics


# Drop missing Topic
before = len(df_task1_final)
df_task1_final = df_task1_final[df_task1_final["Topic"].notna()]
after = len(df_task1_final)

print("Dropped missing Topic:", before - after)

Dropped missing Topic: 860


In [88]:
# Reorder columns
cols_task1 = [
    'Topic','Question','Essay',
    'Task_Response','Coherence_Cohesion',
    'Lexical_Resource','Range_Accuracy',
    'Overall','Examiner_Comment'
]

df_task1_final = df_task1_final[cols_task1]



In [89]:
# Reset index
df_task1_final = df_task1_final.reset_index(drop=True)

print("Final Task 1 size:", len(df_task1_final))

Final Task 1 size: 8548


In [90]:
df_task1_final

,Topic,Question,Essay,Task_Response,Coherence_Cohesion,Lexical_Resource,Range_Accuracy,Overall,Examiner_Comment
0,Bar Chart,(Bar chart) The graphs below show the total pe...,"Firstly, a research is done to show the total ...",NaN,NaN,NaN,NaN,4.5,**Task Achievement: [5]**\nThe response addres...
1,Bar Chart,(Bar chart) The graphs below show the total pe...,The bar chart compares the proportion of films...,NaN,NaN,NaN,NaN,6.0,**Task Achievement: [7]**\nThe report provides...
2,Bar Chart,(Bar chart) The graphs below show the total pe...,The bar charts depict the share of films relea...,NaN,NaN,NaN,NaN,9.0,**Task Achievement: [9]**\nThe report provides...
3,Bar Chart,(Bar chart) The graphs below show the total pe...,The bar charts depicts the number of movies in...,NaN,NaN,NaN,NaN,5.0,**Task Achievement: [6]**\nThe report provides...
4,Bar Chart,(Bar chart) The graphs below show the total pe...,The bar charts display the whole percentage of...,NaN,NaN,NaN,NaN,6.5,**Task Achievement: [7]**\nThe main features o...
...,...,...,...,...,...,...,...,...,...
8543,Bar Chart,"those aged from 18-30 and those aged 45-60, an...",The results of a survey conducted by a personn...,NaN,NaN,NaN,NaN,4.5,**Task Achievement: [5]**\nThe report provides...
8544,Bar Chart,"those aged from 18-30 and those aged 45-60, an...",The stipulated vertical bar chart depicts the ...,NaN,NaN,NaN,NaN,4.0,**Task Achievement: [4]**\nThe report does not...
8545,Bar Chart,"those aged from 18-30 and those aged 45-60, an...",The supplied bar graph compares different fact...,NaN,NaN,NaN,NaN,5.0,**Task Achievement: [6]**\nThe report addresse...
8546,Bar Chart,"those aged from 18-30 and those aged 45-60, an...",The supplied bar graph compares various factor...,NaN,NaN,NaN,NaN,8.0,**Task Achievement: [7]**\nThe report is well-...


In [91]:
df1_task2 = df_task1[df_task1["Task_Type"] == 2].copy()
df2_task2 = df2_task1[df2_task1["Task_Type"] == 2].copy()

In [92]:
df1_task2_clean = pd.DataFrame({
    "Question": df1_task2["Question"],
    "Essay": df1_task2["Essay"],
    "Task_Achievement": df1_task2["Task_Response"],
    "Coherence_Cohesion": df1_task2["Coherence_Cohesion"],
    "Lexical_Resource": df1_task2["Lexical_Resource"],
    "Range_Accuracy": df1_task2["Range_Accuracy"],
    "Overall": df1_task2["Overall"],
    "Examiner_Comment": df1_task2["Examiner_Commen"]
})

df2_task2_clean = pd.DataFrame({
    "Question": df2_task2["Question"],
    "Essay": df2_task2["Essay"],
    "Task_Achievement": np.nan,
    "Coherence_Cohesion": np.nan,
    "Lexical_Resource": np.nan,
    "Range_Accuracy": np.nan,
    "Overall": df2_task2["Overall_Score"],
    "Examiner_Comment": df2_task2["Examiner_Comments"]
})

In [93]:
df_task2_final = pd.concat([
    df1_task2_clean,
    df2_task2_clean
], ignore_index=True)


In [94]:
essay_q_count2 = df_task2_final.groupby("Essay")["Question"].nunique()

conflict_essays2 = essay_q_count2[essay_q_count2 > 1].index

df_task2_final = df_task2_final[
    ~df_task2_final["Essay"].isin(conflict_essays2)
]

print("Task 2 - Removed cross-question essays:", len(conflict_essays2))

Task 2 - Removed cross-question essays: 0


In [95]:
grouped2 = df_task2_final.groupby(["Question", "Essay"]).agg({
    "Overall": list,
    "Task_Achievement": "first",
    "Coherence_Cohesion": "first",
    "Lexical_Resource": "first",
    "Range_Accuracy": "first",
    "Examiner_Comment": "first"
}).reset_index()

In [96]:
grouped2["Overall_clean"] = grouped2["Overall"].apply(resolve_scores)

before = len(grouped2)
grouped2 = grouped2.dropna(subset=["Overall_clean"])
after = len(grouped2)

print("Task 2 - Removed noisy:", before - after)

Task 2 - Removed noisy: 0


In [97]:
df_task2_final = grouped2.drop(columns=["Overall"])
df_task2_final = df_task2_final.rename(columns={"Overall_clean": "Overall"})
df_task2_final["Overall"] = df_task2_final["Overall"].apply(round_band)

In [98]:

cols_task2 = [
    'Question','Essay',
    'Task_Achievement','Coherence_Cohesion',
    'Lexical_Resource','Range_Accuracy',
    'Overall','Examiner_Comment'
]

df_task2_final = df_task2_final[cols_task2]

In [99]:
# Reset index
df_task2_final = df_task2_final.reset_index(drop=True)

print("Final Task 2 size:", len(df_task2_final))

Final Task 2 size: 711


In [100]:
df_task2_final

,Question,Essay,Task_Achievement,Coherence_Cohesion,Lexical_Resource,Range_Accuracy,Overall,Examiner_Comment
0,A country becomes more interesting and develop...,A nation becomes more interesting and advance ...,NaN,NaN,NaN,NaN,7.0,None
1,A country becomes more interesting and develop...,It is belived that countries are developing fa...,NaN,NaN,NaN,NaN,6.0,None
2,A country becomes more interesting and develop...,Some people believe that having a population w...,NaN,NaN,NaN,NaN,7.5,None
3,A country becomes more interesting and develop...,There are people who believe that the mixture ...,NaN,NaN,NaN,NaN,6.5,None
4,A country becomes more interesting and develop...,There are some beliefs that when there is a mi...,NaN,NaN,NaN,NaN,7.5,None
...,...,...,...,...,...,...,...,...
706,should unpaid community work be mandatory in h...,Some people believe that unpaid community serv...,NaN,NaN,NaN,NaN,8.0,None
707,« Previous,Nowadays celebrities earn more money than poli...,NaN,NaN,NaN,NaN,8.0,None
708,‘Failure is proof that the desire wasn’t stron...,"Failure is what nobody wish for, but it could ...",NaN,NaN,NaN,NaN,7.0,None
709,‘Failure is proof that the desire wasn’t stron...,"Generally, in our real lives, failure and succ...",NaN,NaN,NaN,NaN,6.5,None


In [101]:
df_task1_final.to_csv('../data/raw_split/task1_merged.csv', index=False)
df_task2_final.to_csv('../data/raw_split/task2_merged.csv', index=False)